In [60]:
import os
import json
# Step 1: Get all iReal song file names and parse out the song name
ireal_dir = '../dataset/ireal_songs'
ireal_files = [f for f in os.listdir(ireal_dir) if f.endswith('.json')]

# filename pattern: '0000_S_Wonderfil.json'
def extract_ireal_song_name(filename):
    name_part = filename.split('_', 1)[-1]
    name_part = name_part.rsplit('.', 1)[0]
    return name_part

ireal_song_names = [extract_ireal_song_name(f) for f in ireal_files]
print(f"Loaded {len(ireal_song_names)} iReal song names.")


Loaded 4007 iReal song names.


In [61]:
def parse_ireal_json(filepath):
    """
    Parse an iReal Pro JSON file, returning a list of valid chord events with their start (and computed end) offsets,
    and the metadata.
    Returns: (chord_events, metadata)
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    chords = data['chords']
    offsets = data['offsets']
    metadata = data['metadata']  # Extract metadata

    ignore_tokens = {'<start>', '<end>', '<pad>', '.', '|', 
                     'Repeat_0', 'Repeat_1', 'Repeat_2', 'Repeat_3', 
                     'Form_A', 'Form_B', 'Form_C', 'Form_D', 
                     'Form_verse', 'Form_intro', 'Form_Coda', 
                     'Form_Segno', '|:', ':|'}
    ignore_tokens.update([c for c in chords if c.startswith('Form_')])

    # Extract only chord events with valid offsets
    chord_events = []
    for chord, offset in zip(chords, offsets):
        if chord not in ignore_tokens:
            chord_events.append({'chord': chord, 'start': offset})

    # Add end times: end is next start, or +2 beats if last
    for i, event in enumerate(chord_events):
        if i + 1 < len(chord_events):
            event['end'] = chord_events[i + 1]['start']
        else:
            event['end'] = event['start'] + 2

    return chord_events, metadata

In [62]:
from music21 import stream, chord, key, harmony 

def is_true_trigram(trigram):
    # True trigram: all chords must be different
    return len(set(trigram)) == 3

def get_functions(trigram):
    """
    Map each chord in trigram to its functional label in the estimated tonality.
    """
    s = stream.Stream()
    func = []
    for chord in trigram:
        if chord.find('b') > 0 and chord[1:2] == 'b':
            chord = chord[0:1] + '-' + chord[2:]
        #print(chord)
        d = harmony.ChordSymbol(chord)
        d.quarterLength = 2
        s.append(d)
    
    key = s.analyze('key')
    myKey = f"{key.tonic.name} {key.mode}"
    
    # Replace - with b in the key name if needed
    myKey = myKey.replace('-', 'b')
    
    # separate key name and mode
    for chord in trigram:
        if chord.find('b') > 0 and chord[1:2] == 'b':
            chord = chord[0:1] + '-' + chord[2:]
        h = harmony.ChordSymbol(chord)
        h.key = key
        roman_numeral = h.romanNumeral.figure
        # Replace - with b in roman numeral if needed
        roman_numeral = roman_numeral.replace('-', 'b')
        func.append(roman_numeral)
    
    dict = {'tonality': myKey, 'function': func}
    return dict

In [ ]:
import os
import json
from collections import defaultdict
from tqdm import tqdm

ireal_dir = '../dataset/ireal_songs'
ireal_files = [f for f in os.listdir(ireal_dir) if f.endswith('.json')]

def extract_song_id(filename):
    """Extract the 4-digit ID from filename like '0000_S_Wonderful.json'"""
    return filename[:4]

trigram_counts = defaultdict(lambda: {
    "function": None,
    "tonality": None,
    "song_ids": [],
    "styles": [],
    "composers": [],
    "count": 0
})

subset = ireal_files[:40]
for ireal_file in tqdm(ireal_files, desc="Processing iReal songs"):
    filepath = os.path.join(ireal_dir, ireal_file)
    chord_events, metadata = parse_ireal_json(filepath)
    chord_seq = [event['chord'] for event in chord_events]
    song_id = extract_song_id(ireal_file)
    
    if len(chord_seq) < 3:
        continue

    for i in range(len(chord_seq) - 2):
        trigram = tuple(chord_seq[i:i+3])
        if not is_true_trigram(trigram):
            continue
        function = get_functions(trigram)

        # Use only trigram, function, and tonality as key
        key = (trigram, tuple(function["function"]), function["tonality"])
    
        # Initialize if first time
        if trigram_counts[key]["function"] is None:
            trigram_counts[key]["function"] = function["function"]
            trigram_counts[key]["tonality"] = function["tonality"]
        
        # Add song info to arrays (avoid duplicates)
        if song_id not in trigram_counts[key]["song_ids"]:
            trigram_counts[key]["song_ids"].append(song_id)
        
        if metadata["style"] not in trigram_counts[key]["styles"]:
            trigram_counts[key]["styles"].append(metadata["style"])
            
        if metadata["composer"] not in trigram_counts[key]["composers"]:
            trigram_counts[key]["composers"].append(metadata["composer"])
        
        trigram_counts[key]["count"] += 1

# Convert to output list
output_list = []
for (trigram, function, tonality), value in trigram_counts.items():
    output_list.append({
        "trigram": list(trigram),
        "function": list(function),
        "tonality": tonality,
        "song_ids": value["song_ids"],
        "styles": value["styles"],
        "composers": value["composers"],
        "count": value["count"]
    })

# Sort by count, descending
output_list.sort(key=lambda x: x["count"], reverse=True)

output_dir = "/workspace/dataset/trigrams"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "trigram_dataset.json")
with open(output_path, "w") as f:
    json.dump(output_list, f, indent=2)
print(f"Saved sorted trigrams to {output_path}")